In [1]:
from collections import defaultdict, Counter

class FiveGramLanguageModel:

    def __init__(self, discount=0.75):

        self.discount = discount

        self.vocab = set()

        self.ngrams = {
            1: Counter(),
            2: defaultdict(Counter),
            3: defaultdict(Counter),
            4: defaultdict(Counter),
            5: defaultdict(Counter)
        }

    def train(self, corpus):

        for sentence in corpus:

            words = sentence.lower().split()

            words = ["<s>"]*4 + words + ["</s>"]

            for word in words:
                self.vocab.add(word)

            for i in range(len(words)):

                self.ngrams[1][words[i]] += 1

                if i >= 1:
                    self.ngrams[2][tuple(words[i-1:i])][words[i]] += 1

                if i >= 2:
                    self.ngrams[3][tuple(words[i-2:i])][words[i]] += 1

                if i >= 3:
                    self.ngrams[4][tuple(words[i-3:i])][words[i]] += 1

                if i >= 4:
                    self.ngrams[5][tuple(words[i-4:i])][words[i]] += 1

    # Recursive Kneser-Ney Style

    def probability(self, history, word):

        order = len(history)+1

        if order == 1:

            total = sum(self.ngrams[1].values())

            return self.ngrams[1][word]/total

        history = tuple(history)

        counts = self.ngrams[order][history]

        total = sum(counts.values())

        if total == 0:

            return self.probability(history[1:], word)

        c = counts[word]

        discounted = max(c-self.discount,0)/total

        backoff = self.discount*len(counts)/total

        return discounted + backoff*self.probability(history[1:],word)

    # Prediction

    def predict(self, history, top=5):

        history = history.lower().split()

        history = ["<s>"]*4 + history

        history = history[-4:]

        scores = {}

        for word in self.vocab:

            if word in ["<s>","</s>"]:
                continue

            scores[word] = self.probability(history,word)

        result = sorted(scores.items(),
                        key=lambda x:x[1],
                        reverse=True)

        return result[:top]

# Training Data

queries = [

"best places to visit in india",

"best places to visit in chennai",

"best places to visit near me",

"best places to visit during summer",

"best restaurants near me",

"best hotels in chennai",

"artificial intelligence course",

"machine learning tutorial"

]

model = FiveGramLanguageModel()

model.train(queries)

# User Query

query = input("Enter Search Query: ")

suggestions = model.predict(query)

print("\nAutocomplete Suggestions:\n")

for word,score in suggestions:

    print(word,"->",round(score,4))

Enter Search Query:  what is 



Autocomplete Suggestions:

best -> 0.0769
to -> 0.0513
visit -> 0.0513
places -> 0.0513
in -> 0.0385
